# 02_Analysis.ipynb — Analysis
Computes sinuosity, slope, segment means, flux, fits beta'1 (slope profile)
and beta'2 (elevation profile) with SE, plus goodness-of-fit for both.

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats as scipy_stats

OUTPUT_DIR = "Output/"


## Load data

In [149]:
with open("data.pkl", "rb") as f:
    data = pickle.load(f)
stream_registry = data["stream_registry"]
ALPHA           = data["ALPHA"]
print(f"Loaded {len(stream_registry)} streams")


Loaded 16 streams


## Step 1 — Sinuosity (arc & chord length per segment) 
   #### Refer Supplementary Section 1

In [150]:
def compute_segment_stats(df):
    rows = []

    for seg_id, g in df.groupby("segment_id"):

        if len(g) < 2:
            continue

        xy = g[["x", "y"]].values

        # Straight-line length
        normal_len = np.linalg.norm(xy[-1] - xy[0])

        # Stream length
        sinuous_len = np.sum(
            np.linalg.norm(np.diff(xy, axis=0), axis=1)
        )

        rows.append([
            seg_id,
            normal_len,
            sinuous_len,
        ])

    ss = pd.DataFrame(
        rows,
        columns=[
            "segment_id",
            "normal_length",
            "sinuous_length",
        ],
    )

    # -------------------------------------------------------
    # Cumulative distances
    # -------------------------------------------------------

    ss["Acc_Norm"] = ss["normal_length"].cumsum()
    ss["Acc_Sin"] = ss["sinuous_length"].cumsum()

    # Midpoint cumulative distances
    ss["Acc_Norm_Mean"] = (
        ss["Acc_Norm"] - ss["normal_length"] / 2
    )

    ss["Acc_Sin_Mean"] = (
        ss["Acc_Sin"] - ss["sinuous_length"] / 2
    )

    ss["sinuosity"] = ss["sinuous_length"] / ss["normal_length"]

    return ss


for entry in stream_registry:
    df = entry[0]
    df.attrs["seg_stats"] = compute_segment_stats(df)

print("Sinuosity computed.")

Sinuosity computed.


## Step 2 — Segment means (velocity, elevation, relief, x, y, distance)
   ### Refer supplementary text S1

In [151]:
def compute_segment_means(df):
    vc     = df.attrs.get("vel_col",  "velocity")
    ec     = df.attrs.get("elev_col", "elev")
    offset = df.attrs.get("elev_offset", 0)

    agg = (
        df.groupby("segment_id")
          .agg(relief         = ("relief",   "mean"),
               velocity       = (vc,         "mean"),
               x              = ("x",        "mean"),
               y              = ("y",        "mean"),
               Distance       = ("dist_N",   "mean"),
               elevation      = (ec,         "mean"),
               elevation_max  = (ec,         "max"),
               )
          .reset_index()
    )

    agg["elevation"] = agg["elevation"] + offset

    return agg

for entry in stream_registry:
    df = entry[0]
    df.attrs["seg_means"] = compute_segment_means(df)

print("Segment means computed.")

Segment means computed.


## Step 3 — Slope per segment

#### See supplementtary text S1

In [152]:
def compute_slope(df, seg_stats, seg_means, elev_col="elev"):
    # ----------------------------------------------------------
    # Match lengths
    # ----------------------------------------------------------
    n_rows = min(len(seg_stats), len(seg_means))

    ss = seg_stats.iloc[:n_rows].copy().reset_index(drop=True)
    sm = seg_means.iloc[:n_rows].copy().reset_index(drop=True)

    # ----------------------------------------------------------
    # Common elevation values
    # ----------------------------------------------------------
    elev = sm["elevation"].values

    # ==========================================================
    # slope_l
    # Use cumulative stream distance (segment centres)
    # ==========================================================

    acc_sin = ss["Acc_Sin_Mean"].values

    delta_elev = np.diff(elev)
    delta_dist = np.diff(acc_sin)

    with np.errstate(divide="ignore", invalid="ignore"):
        slope_l = -delta_elev / delta_dist

    slope_l = np.concatenate([[np.nan], slope_l])

    ss["slope_l"] = slope_l

    # ==========================================================
    # slope_d
    # Use cumulative normal distance (segment centres)
    # ==========================================================

    acc_norm = ss["Acc_Norm_Mean"].values

    delta_elev = np.diff(elev)
    delta_dist = np.diff(acc_norm)

    with np.errstate(divide="ignore", invalid="ignore"):
        slope_d = -delta_elev / delta_dist

    slope_d = np.concatenate([[np.nan], slope_d])

    ss["slope_d"] = slope_d

    return ss


# ----------------------------------------------------------
# Compute for every stream
# ----------------------------------------------------------

for entry in stream_registry:

    df = entry[0]

    ec = df.attrs.get("elev_col", "elev")

    df.attrs["seg_stats"] = compute_slope(
        df,
        df.attrs["seg_stats"],
        df.attrs["seg_means"],
        elev_col=ec,
    )

print("Slopes computed.")

Slopes computed.


## Step 4 — Merge segment means into segment stats and compute flux

In [153]:
SEG_LEN_MAP = {
    "sia_1": 2000, "sia_2": 2000, "sia_3": 2000,
    "sia_4": 2000, "sia_5": 2000, "sia_6": 2000,
    "ron_1": 2000, "vet_1": 2000,
    "fou_1": 500,  "fou_2": 500,  "fou_3": 500,  "fou_4": 500,
    "von_1": 500,  "von_2": 500,  "von_3": 500,  "von_4": 500,
}

def compute_flux(df, seg_stats, seg_means, label):
    sm = seg_means.copy()

    # columns to merge from seg_means
    merge_cols = [c for c in ["segment_id", "velocity", "elevation", "relief",
                               "x", "y", "Distance", "elevation_max"]
                  if c in sm.columns]

    # drop conflicting columns to avoid _x/_y suffix on re-run
    cols_to_drop = [c for c in merge_cols
                    if c != "segment_id" and c in seg_stats.columns]
    ss = seg_stats.drop(columns=cols_to_drop, errors="ignore").copy()
    ss = ss.merge(sm[merge_cols], on="segment_id", how="left")

    seg_len = SEG_LEN_MAP.get(label, 2000)

    ss["flux"]      = ss["sinuosity"] * ss["velocity"]

    # ── flux of stream length ─────────────────────────────────────────
    # ((V1 * dx1) - (V2 * dx2)) / (dx2 * dx'2)
    # V1, dx1  = velocity and Acc_Sin_Mean  of previous segment (i-1)
    # V2, dx2  = velocity and Acc_Sin_Mean  of current  segment (i)
    # dx'2     = Acc_Norm_Mean of current segment (i)

    V   = ss["velocity"].values
    dx  = ss["sinuous_length"].values
    dxn = ss["normal_length"].values

    V1  = np.concatenate([[np.nan], V[:-1]])    # previous segment velocity
    dx1 = np.concatenate([[np.nan], dx[:-1]])   # previous segment Acc_Sin_Mean

    with np.errstate(divide="ignore", invalid="ignore"):
        flux_sl = ((V1 * dx1) - (V * dx)) / (dx * dxn)

    ss["flux_stream_len"] = flux_sl             # first segment = NaN

    return ss

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    df.attrs["seg_stats"] = compute_flux(df, df.attrs["seg_stats"],
                                         df.attrs["seg_means"], label)

print("Flux computed and merged into seg_stats.")

Flux computed and merged into seg_stats.


## Step 5 — Remove last segment row (Stream length <500/2000m)

In [154]:
for entry in stream_registry:
    df, group, label, *_ = entry
    df.attrs["seg_stats"] = df.attrs["seg_stats"].iloc[:-1].reset_index(drop=True)

print("Last row removed.")

Last row removed.


## Step 6 — Find beta'1 from slope profile
   #### Refer Eq. (3)

In [155]:
def fit_gamma1(x, slope, alpha):
    """Fit beta'_1 from slope profile S = beta'_1 * x^((alpha-1)/2)."""
    exp = (alpha - 1) / 2

    if alpha == 1:           # constant: beta'_1 = mean(S)
        b1   = np.mean(slope)
        n    = len(slope)
        se   = np.std(slope, ddof=1) / np.sqrt(n)
        pred = np.full_like(slope, b1, dtype=float)
    else:                    # power-law through origin
        X    = x ** exp
        b1   = np.sum(X * slope) / np.sum(X ** 2)
        resid = slope - b1 * X
        n    = len(slope)
        se   = np.sqrt(np.sum(resid**2) / ((n - 1) * np.sum(X**2)))
        pred = b1 * X

    return b1, se, pred

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    alpha = ALPHA[trend]
    ss    = df.attrs["seg_stats"].dropna(subset=["slope_l"])
    x, sl = ss["Distance"].values, ss["slope_l"].values

    g1, se1, sl_pred = fit_gamma1(x, sl, alpha)
    df.attrs["beta'1"]     = g1
    df.attrs["SE_beta'1"]  = se1
    df.attrs["slope_pred"] = sl_pred
    df.attrs["slope_x"]    = x
    df.attrs["slope_obs"]  = sl
    df.attrs["alpha"]      = alpha

print(f"{'Stream':18s}  {"beta'_1":>12s}  {"SE_beta'1":>12s}")
print("-" * 48)
for entry in stream_registry:
    df, group, label, *_ = entry
    print(f"{label:18s}  {df.attrs["beta'1"]:12.4g}  {df.attrs["SE_beta'1"]:12.4g}")


Stream                   beta'_1     SE_beta'1
------------------------------------------------
sia_1                      1.386        0.1771
sia_2                    0.01963      0.001031
sia_3                      1.495        0.1235
sia_4                    0.01887      0.001044
sia_5                    0.01582     0.0007739
sia_6                      1.542        0.1216
ron_1                       2.91        0.1944
fou_1                  0.0009678      5.79e-05
fou_2                   0.001107     8.715e-05
fou_3                   0.001303     8.185e-05
fou_4                   0.001602     0.0001442
von_1                   0.000577     2.532e-05
von_2                  0.0006421     2.384e-05
von_3                  0.0007528     3.767e-05
von_4                  0.0006072     2.806e-05
vet_1                    0.02305     0.0009999


## Step 7 — Find beta'2 from elevation profile
   #### Refer Eq. (4)

In [156]:
def fit_beta2(x, z, alpha):
    """Fit beta'_2 from elevation profile."""
    exp = (alpha + 1) / 2
    x0, z0 = x[0], z[0]
    F  = x**exp - x0**exp
    Y  = z0 - z

    k  = np.sum(F * Y) / np.sum(F**2)
    b2 = k * exp

    resid = Y - k * F
    n     = len(Y)
    se_k  = np.sqrt(np.sum(resid**2) / ((n - 1) * np.sum(F**2)))
    se_b2 = se_k * exp

    z_pred = z0 - k * F
    return b2, se_b2, z_pred

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    alpha = ALPHA[trend]
    ss    = df.attrs["seg_stats"]
    x     = ss["Distance"].values
    z     = ss["elevation"].values   # already merged into seg_stats

    b2, se2, z_pred = fit_beta2(x, z, alpha)

    # Veteranbreen: correct z_pred values below 0 (offset artifact)
    if group == "Veteranbreen":
        z_pred = np.where(z_pred < 0, z_pred + 2000, z_pred)

    df.attrs["beta'2"]    = b2
    df.attrs["SE_beta'2"] = se2
    df.attrs["z_pred"]    = z_pred
    df.attrs["elev_x"]    = x
    df.attrs["elev_obs"]  = z

print(f"{'Stream':18s}  {"beta'2":>12s}  {"SE_beta'2":>12s}")
print("-" * 48)
for entry in stream_registry:
    df, group, label, *_ = entry
    print(f"{label:18s}  {df.attrs["beta'2"]:12.4g}  {df.attrs["SE_beta'2"]:12.4g}")


Stream                    beta'2     SE_beta'2
------------------------------------------------
sia_1                       1.21        0.0498
sia_2                    0.01927     0.0001698
sia_3                      1.291       0.04236
sia_4                    0.01837     9.311e-05
sia_5                    0.01567     9.759e-05
sia_6                      1.381       0.02196
ron_1                      2.506       0.02742
fou_1                   0.001034     1.571e-05
fou_2                   0.001279     3.846e-05
fou_3                   0.001689     4.965e-05
fou_4                   0.001781     5.314e-05
von_1                   0.000648     8.922e-06
von_2                  0.0006479     5.523e-06
von_3                  0.0008513     1.435e-05
von_4                   0.000641      7.72e-06
vet_1                    0.02201     0.0002678


## Step 8 — Goodness-of-fit: slope profile (beta'1)
 #### See Supplementary Table S3

In [157]:
print("\n=== Slope Profile Fit (beta'1) — per stream ===")
print(f"{'Stream':18s}  {'R²':>7s}  {'Adj R²':>8s}  "
      f"{'RMSE':>10s}  {'NRMSE(M)':>10s}  {'NRMSE(R)':>10s}   {'Pearson r':>10s}")
print("-" * 95)

all_obs_s, all_pred_s = [], []
all_nrmse_R = []    # collect NRMSE(R) per stream for median and IQR
k = 1

for entry in stream_registry:
    df, group, label, *_ = entry
    obs  = df.attrs["slope_obs"]
    pred = df.attrs["slope_pred"]
    n    = min(len(obs), len(pred))
    o, p = obs[:n], pred[:n]
    mask = np.isfinite(o) & np.isfinite(p)
    o, p = o[mask], p[mask]
    if len(o) < 2:
        continue

    r2       = r2_score(o, p)
    adj_r2   = 1 - (1 - r2) * (len(o) - 1) / (len(o) - k - 1)
    rmse     = np.sqrt(mean_squared_error(o, p))
    nrmse_M  = rmse / np.mean(o)
    nrmse_R  = rmse / (np.max(o) - np.min(o))
    r, _     = scipy_stats.pearsonr(o, p)

    all_nrmse_R.append(nrmse_R)    # store for median and IQR

    print(f"{label:18s}  "
          f"{r2:7.4f}  "
          f"{adj_r2:8.4f}  "
          f"{rmse:10.6f}  "
          f"{nrmse_M:10.6f}  "
          f"{nrmse_R:10.6f}  "
          f"{r:10.4f}")

    all_obs_s.extend(o)
    all_pred_s.extend(p)

# ── pooled statistics ────────────────────────────────────────────────
ao, ap = np.array(all_obs_s), np.array(all_pred_s)
mask   = np.isfinite(ao) & np.isfinite(ap)
ao, ap = ao[mask], ap[mask]

r2p      = r2_score(ao, ap)
adj_r2p  = 1 - (1 - r2p) * (len(ao) - 1) / (len(ao) - k - 1)
rmsep    = np.sqrt(mean_squared_error(ao, ap))
nrmsep_M = rmsep / np.mean(ao)
nrmsep_R = rmsep / (np.max(ao) - np.min(ao))
rp, _    = scipy_stats.pearsonr(ao, ap)

# ── median and IQR of NRMSE(R) across all streams ───────────────────
all_nrmse_R  = np.array(all_nrmse_R)
median_nrmse_R = np.median(all_nrmse_R)
q75, q25       = np.percentile(all_nrmse_R, [75, 25])
iqr_nrmse_R    = q75 - q25

print("-" * 95)
print(f"{'POOLED (all)':18s}  "
      f"{r2p:7.4f}  "
      f"{adj_r2p:8.4f}  "
      f"{rmsep:10.6f}  "
      f"{nrmsep_M:10.6f}  "
      f"{nrmsep_R:10.6f}  "
      f"{rp:10.4f}")

print("\n--- NRMSE(R) summary across all streams ---")
print(f"  {'Median NRMSE(R)':<20s}  {median_nrmse_R:.6f}")
print(f"  {'IQR NRMSE(R)':<20s}  {iqr_nrmse_R:.6f}")
print(f"  {'Q25':<20s}  {q25:.6f}")
print(f"  {'Q75':<20s}  {q75:.6f}")


=== Slope Profile Fit (beta'1) — per stream ===
Stream                   R²    Adj R²        RMSE    NRMSE(M)    NRMSE(R)    Pearson r
-----------------------------------------------------------------------------------------------
sia_1               -0.6122   -1.1497    0.004696    0.252474    0.509655      0.3357
sia_2                0.0000   -0.0667    0.004124    0.210119    0.278155         nan
sia_3               -0.3141   -0.7521    0.003274    0.164621    0.417167      0.7290
sia_4                0.0000   -0.0500    0.004783    0.253517    0.223234         nan
sia_5                0.0000   -0.0476    0.003630    0.229465    0.229850         nan
sia_6                0.6254    0.5719    0.003856    0.232936    0.194169      0.7916
ron_1                0.7510    0.7154    0.006163    0.200026    0.135126      0.8702
fou_1                0.5646    0.5162    0.010441    0.193737    0.246116      0.7673
fou_2               -0.5452   -0.7659    0.012919    0.220710    0.359054      0

-----------------------------------------------------------------------------------------------
POOLED (all)         0.8563    0.8555    0.006903    0.209239    0.070510      0.9269

--- NRMSE(R) summary across all streams ---
  Median NRMSE(R)       0.237983
  IQR NRMSE(R)          0.083571
  Q25                   0.198953
  Q75                   0.282523


/tmp/ipykernel_3391148/1247299672.py:26: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _     = scipy_stats.pearsonr(o, p)
/tmp/ipykernel_3391148/1247299672.py:26: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _     = scipy_stats.pearsonr(o, p)
/tmp/ipykernel_3391148/1247299672.py:26: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _     = scipy_stats.pearsonr(o, p)
/tmp/ipykernel_3391148/1247299672.py:26: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _     = scipy_stats.pearsonr(o, p)


## Step 9 — Goodness-of-fit: elevation profile (beta'2)
   #### See Supplementary Table S3

In [158]:
print("\n Elevation Profile Fit (beta'2) — per stream")
print(f"{'Stream':18s}  {'R²':>7s}  {'Adj R²':>8s}  "
      f"{'RMSE(m)':>10s}  {'NRMSE(M)':>10s}  {'NRMSE(R)':>10s}  "
      f"{'Pearson r':>10s}")
print("-" * 95)

all_obs_e, all_pred_e = [], []
all_nrmse_R = []    # collect NRMSE(R) per stream for median and IQR
k = 1

for entry in stream_registry:
    df, group, label, *_ = entry
    obs  = df.attrs["elev_obs"]
    pred = df.attrs["z_pred"]
    n    = min(len(obs), len(pred))
    o, p = obs[:n], pred[:n]
    mask = np.isfinite(o) & np.isfinite(p)
    o, p = o[mask], p[mask]
    if len(o) < 2:
        continue

    r2       = r2_score(o, p)
    adj_r2   = 1 - (1 - r2) * (len(o) - 1) / (len(o) - k - 1)
    rmse     = np.sqrt(mean_squared_error(o, p))
    nrmse_M  = rmse / np.mean(o)
    nrmse_R  = rmse / (np.max(o) - np.min(o))
    r, _     = scipy_stats.pearsonr(o, p)

    all_nrmse_R.append(nrmse_R)    # store for median and IQR

    print(f"{label:18s}  "
          f"{r2:7.4f}  "
          f"{adj_r2:8.4f}  "
          f"{rmse:10.2f}  "
          f"{nrmse_M:10.4f}  "
          f"{nrmse_R:10.4f}  "
          f"{r:10.4f}")

    all_obs_e.extend(o)
    all_pred_e.extend(p)

# ── pooled statistics ────────────────────────────────────────────────
ao, ap = np.array(all_obs_e), np.array(all_pred_e)
mask   = np.isfinite(ao) & np.isfinite(ap)
ao, ap = ao[mask], ap[mask]

r2p      = r2_score(ao, ap)
adj_r2p  = 1 - (1 - r2p) * (len(ao) - 1) / (len(ao) - k - 1)
rmsep    = np.sqrt(mean_squared_error(ao, ap))
nrmsep_M = rmsep / np.mean(ao)
nrmsep_R = rmsep / (np.max(ao) - np.min(ao))
rp, _    = scipy_stats.pearsonr(ao, ap)

# ── median and IQR of NRMSE(R) across all streams ───────────────────
all_nrmse_R    = np.array(all_nrmse_R)
median_nrmse_R = np.median(all_nrmse_R)
q75, q25       = np.percentile(all_nrmse_R, [75, 25])
iqr_nrmse_R    = q75 - q25

print("-" * 95)
print(f"{'POOLED (all)':18s}  "
      f"{r2p:7.4f}  "
      f"{adj_r2p:8.4f}  "
      f"{rmsep:10.2f}  "
      f"{nrmsep_M:10.4f}  "
      f"{nrmsep_R:10.4f}  "
      f"{rp:10.4f}")

print("\n--- NRMSE(R) summary across all streams ---")
print(f"  {'Median NRMSE(R)':<20s}  {median_nrmse_R:.4f}")
print(f"  {'IQR NRMSE(R)':<20s}  {iqr_nrmse_R:.4f}")
print(f"  {'Q25':<20s}  {q25:.4f}")
print(f"  {'Q75':<20s}  {q75:.4f}")


 Elevation Profile Fit (beta'2) — per stream
Stream                   R²    Adj R²     RMSE(m)    NRMSE(M)    NRMSE(R)   Pearson r
-----------------------------------------------------------------------------------------------
sia_1                0.9730    0.9662       10.80      0.0023      0.0582      0.9911
sia_2                0.9953    0.9950       13.94      0.0031      0.0209      0.9978
sia_3                0.9821    0.9777        9.20      0.0019      0.0464      0.9942
sia_4                0.9978    0.9977       11.21      0.0025      0.0135      0.9992
sia_5                0.9965    0.9963       12.57      0.0028      0.0173      0.9988
sia_6                0.9903    0.9891        9.35      0.0020      0.0314      0.9967
ron_1                0.9950    0.9944       11.66      0.0021      0.0211      0.9976
fou_1                0.9924    0.9917        7.95      0.0150      0.0270      0.9983
fou_2                0.9733    0.9700       13.34      0.0282      0.0510      0.995

## Step 10 — Predicted sinuosity (using beta'1)
   #### Refer Section 5.2 Eq. (6)

In [159]:
for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    ss     = df.attrs["seg_stats"]
    x      = ss["Distance"].values
    slope  = ss["slope_d"].values
    alpha  = df.attrs["alpha"]
    beta1 = df.attrs["beta'1"]

    exp = (alpha - 1) / 2
    with np.errstate(divide="ignore", invalid="ignore"):
        sin_pred = np.where(x > 0, slope / (beta1 * x**exp), np.nan)
    df.attrs["sin_pred"] = sin_pred

print("Predicted sinuosity computed (using beta'_1).")


Predicted sinuosity computed (using beta'_1).


In [160]:
# ── Mean sinuosity, mean slope_l, IQR sinuosity, IQR slope per stream ─
print(f"{'Stream':18s}  {'Mean Sin':>10s}  {'IQR Sin':>10s}  "
      f"{'Mean Slope':>12s}  {'IQR Slope':>12s}")
print("-" * 70)

rows = []

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    ss = df.attrs["seg_stats"].dropna(subset=["sinuosity", "slope_l"])

    mean_sin   = ss["sinuosity"].mean()
    mean_slope = ss["slope_l"].mean()

    q75_sin, q25_sin     = np.percentile(ss["sinuosity"], [75, 25])
    q75_slope, q25_slope = np.percentile(ss["slope_l"],   [75, 25])

    iqr_sin   = q75_sin   - q25_sin
    iqr_slope = q75_slope - q25_slope

    print(f"{label:18s}  "
          f"{mean_sin:10.4f}  "
          f"{iqr_sin:10.4f}  "
          f"{mean_slope:12.6f}  "
          f"{iqr_slope:12.6f}")

    rows.append({
        "Stream":     label,
        "Mean Sin":   round(mean_sin,   4),
        "IQR Sin":    round(iqr_sin,    4),
        "Mean Slope": round(mean_slope, 6),
        "IQR Slope":  round(iqr_slope,  6),
    })

summary_df = pd.DataFrame(rows)

Stream                Mean Sin     IQR Sin    Mean Slope     IQR Slope
----------------------------------------------------------------------
sia_1                   1.0808      0.0218      0.018599      0.006936
sia_2                   1.2242      0.1911      0.019629      0.005345
sia_3                   1.0618      0.0238      0.019888      0.003970
sia_4                   1.3958      0.5053      0.018867      0.004150
sia_5                   1.5511      0.2798      0.015820      0.003302
sia_6                   1.3050      0.0799      0.016553      0.007849
ron_1                   1.5798      0.1874      0.030812      0.001738


fou_1                   1.1714      0.0558      0.053892      0.029732
fou_2                   1.4103      0.1666      0.058536      0.011846
fou_3                   1.0618      0.0104      0.057772      0.011961
fou_4                   1.1189      0.0171      0.070915      0.009241
von_1                   1.2378      0.2591      0.038193      0.009763
von_2                   1.1018      0.0540      0.042435      0.019939
von_3                   1.1076      0.1368      0.044219      0.010011
von_4                   1.0483      0.0293      0.040803      0.016464
vet_1                   1.0699      0.0304      0.023052      0.003525


## Step 11 — Error propagation / uncertainty analysis
   #### Propagates elevation (σ_z) and channel-width (σ_w) measurement
   #### errors into σ_elev, σ_slope_l, σ_slope_d, σ_slope_model, σ_elev_model

In [ ]:
# ── error sources ─────────────────────────────────────────────────────
ELEV_ERROR  = {**{k: 5 for k in ["sia_1","sia_2","sia_3","sia_4","sia_5","sia_6","ron_1"]},
               **{k: 2 for k in ["fou_1","fou_2","fou_3","fou_4","von_1","von_2","von_3","von_4","vet_1"]}}

WIDTH_ERROR = {"sia_1": 3, "sia_2": 3, "sia_3": 3, "sia_4": 4, "sia_5": 3, "sia_6": 4, "ron_1": 10,
               "fou_1": 3, "fou_2": 5, "fou_3": 2, "fou_4": 2,
               "von_1": 5, "von_2": 5, "von_3": 3, "von_4": 5, "vet_1": 5}

# ══════════════════════════════════════════════════════════════════════
# Error propagation formulas (relative uncertainty method)
# ══════════════════════════════════════════════════════════════════════
#
# σ_Lsin  = sqrt(N_steps) * σ_w
# σ_Lnorm = 2 * σ_w
#
# σ_slope_l     = |slope_l| * sqrt( (σ_z/Δz)² + (σ_Lsin/L_sin)² )
# σ_slope_d     = |slope_d| * sqrt( (σ_z/Δz)² + (σ_Lnorm/L_norm)² )
# σ_slope_model = |S_model| * sqrt( (se1/beta1)² + (σ_Lsin/x_sin)² )
# σ_elev_model  = sqrt( σ_z²
#                      + ((zp-z0)*se2/beta2)²
#                      + ((zp-z0)*σ_Lsin/x_sin)² )

# ══════════════════════════════════════════════════════════════════════
# Main computation
# ══════════════════════════════════════════════════════════════════════
all_rows = []

for df, group, label, color, marker, trend in stream_registry:
    σ_z, σ_w = ELEV_ERROR.get(label, 5), WIDTH_ERROR.get(label, 3)

    ss  = df.attrs["seg_stats"].copy()
    ec  = df.attrs.get("elev_col", "elev")
    alpha, beta1, se1 = df.attrs["alpha"], df.attrs["beta'1"], df.attrs["SE_beta'1"]
    beta2, se2        = df.attrs["beta'2"], df.attrs["SE_beta'2"]
    exp1, exp2        = (alpha - 1) / 2, (alpha + 1) / 2

    for seg_id, seg in df.groupby("segment_id"):
        if seg_id not in ss["segment_id"].values:
            continue

        row_ss = ss[ss["segment_id"] == seg_id].iloc[0]
        z, xy  = seg[ec].values, seg[["x", "y"]].values
        if len(z) < 2:
            continue

        # ── geometry ──────────────────────────────────────────────────
        steps   = np.linalg.norm(np.diff(xy, axis=0), axis=1)
        L_sin, L_norm, N_steps = steps.sum(), np.linalg.norm(xy[-1] - xy[0]), len(steps)
        Δz      = abs(z[0] - z[-1])
        σ_Lsin, σ_Lnorm = np.sqrt(N_steps) * σ_w, 2 * σ_w

        # ── 1. elevation uncertainty ──────────────────────────────────
        σ_elev = σ_z

        # ── 2. slope_l uncertainty ────────────────────────────────────
        slope_l = row_ss["slope_l"]
        σ_slope_l = (abs(slope_l) * np.sqrt((σ_z / Δz) ** 2 + (σ_Lsin / L_sin) ** 2)
                     if np.isfinite(slope_l) and Δz > 0 and L_sin > 0 else np.nan)

        # ── 3. slope_d uncertainty ────────────────────────────────────
        slope_d = row_ss["slope_d"]
        σ_slope_d = (abs(slope_d) * np.sqrt((σ_z / Δz) ** 2 + (σ_Lnorm / L_norm) ** 2)
                     if np.isfinite(slope_d) and Δz > 0 and L_norm > 0 else np.nan)

        # ── 4. modeled slope uncertainty ──────────────────────────────
        x_sin = row_ss["Acc_Sin"]
        if np.isfinite(x_sin) and x_sin > 0 and beta1 != 0:
            S_model = beta1 * (x_sin ** exp1)
            σ_slope_model = abs(S_model) * np.sqrt((se1 / beta1) ** 2 + (σ_Lsin / x_sin) ** 2)
        else:
            σ_slope_model = np.nan

        # ── 5. modeled elevation uncertainty ─────────────────────────
        z_pred, idx_in_ss = df.attrs["z_pred"], list(ss["segment_id"]).index(seg_id)
        if idx_in_ss < len(z_pred) and np.isfinite(z_pred[idx_in_ss]) and beta2 != 0 and x_sin > 0:
            zp, z0 = z_pred[idx_in_ss], ss["elevation"].values[0]
            σ_elev_model = np.sqrt(σ_z ** 2 + ((zp - z0) * se2 / beta2) ** 2 + ((zp - z0) * σ_Lsin / x_sin) ** 2)
        else:
            σ_elev_model = np.nan

        all_rows.append({
            "Stream": label, "Group": group, "Trend": trend, "Segment": seg_id,
            "σ_z (m)": σ_z, "σ_w (m)": σ_w,
            "L_sin (m)": round(L_sin, 2), "σ_L_sin (m)": round(σ_Lsin, 4),
            "L_norm (m)": round(L_norm, 2), "σ_L_norm (m)": round(σ_Lnorm, 4),
            "Δz (m)": round(Δz, 4), "σ_elev (m)": round(σ_elev, 2),
            "slope_l": round(slope_l, 6) if np.isfinite(slope_l) else np.nan,
            "σ_slope_l": round(σ_slope_l, 6) if np.isfinite(σ_slope_l) else np.nan,
            "slope_d": round(slope_d, 6) if np.isfinite(slope_d) else np.nan,
            "σ_slope_d": round(σ_slope_d, 6) if np.isfinite(σ_slope_d) else np.nan,
            "σ_slope_model": round(σ_slope_model, 6) if np.isfinite(σ_slope_model) else np.nan,
            "σ_elev_model (m)": round(σ_elev_model, 4) if np.isfinite(σ_elev_model) else np.nan,
        })

error_df = pd.DataFrame(all_rows)

# ── print per stream ──────────────────────────────────────────────────
pd.set_option("display.max_rows", None); pd.set_option("display.max_columns", None); pd.set_option("display.width", 10000)

for label in error_df["Stream"].unique():
    sub = error_df[error_df["Stream"] == label]
    print(f"\n{'═'*120}\n  Stream: {label}  |  σ_z = ±{sub['σ_z (m)'].iloc[0]}m  |  σ_w = ±{sub['σ_w (m)'].iloc[0]}m\n{'═'*120}")
    print(sub.drop(columns=["Stream","Group","Trend","σ_z (m)","σ_w (m)"]).to_string(index=False))
    print(f"\n  Mean σ_slope_l     = {sub['σ_slope_l'].mean():.6f}")
    print(f"  Mean σ_slope_d     = {sub['σ_slope_d'].mean():.6f}")
    print(f"  Mean σ_slope_model = {sub['σ_slope_model'].mean():.6f}")
    print(f"  Mean σ_elev_model  = {sub['σ_elev_model (m)'].mean():.4f} m")

pd.reset_option("display.max_rows"); pd.reset_option("display.max_columns"); pd.reset_option("display.width")

# ── save CSV ──────────────────────────────────────────────────────────
csv_path = OUTPUT_DIR + "error_propagation.csv"
error_df.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

## Step 12 — Save analysis results

In [ ]:
with open("analysis.pkl", "wb") as f:
    pickle.dump({"stream_registry": stream_registry, "ALPHA": ALPHA, "error_df": error_df}, f)
print("Saved analysis.pkl")

### Print raw dataframe, Segment_Means & Segment_stats

In [162]:
# show all columns without truncation
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

# change the label to whichever stream need to display
target_label = "sia_5"

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    if label == target_label:
        print(f"Raw dataframe for {label} (first 5 rows)") # To dispaly all the rows remove head(5).
        print(df.head(5))

        print(f"\nseg_stats for {label} (first 5 rows)")
        print(df.attrs["seg_stats"].head(5))

        print(f"\nseg_means for {label} (first 5 rows)")
        print(df.attrs["seg_means"].head(5))
        break

Raw dataframe for sia_5 (first 5 rows)
   vertex_index   distance       angle      elev            x           y  velocity  relief        dist_N  segment_id
0             0   0.000000  330.785860  4097.321 -1479336.756  766051.002   162.543  19.723  48213.329780          24
1             1   4.740077  330.785860  4097.321 -1479339.070  766055.139   162.543  19.723  48208.589703          24
2             2   9.480154  326.572051  4098.435 -1479341.383  766059.276   162.543  21.599  48203.849626          24
3             3  13.093999  322.358243  4098.435 -1479343.590  766062.138   162.543  21.599  48200.235780          24
4             4  16.707845  340.364046  4099.185 -1479345.797  766065.000   162.543  21.599  48196.621935          24

seg_stats for sia_5 (first 5 rows)
   segment_id  normal_length  sinuous_length     Acc_Norm      Acc_Sin  Acc_Norm_Mean  Acc_Sin_Mean  sinuosity   slope_l   slope_d    velocity    elevation     relief             x              y     Distance  elevati